# Week 5: классификация спутниковых снимков

**Как вести эксперименты**
1. Меняете только словарь `CFG` (раздел 2): имя, заметку «что поменяли» и нужные параметры.
2. Запускаете ноутбук целиком (*Run All*) — так каждый прогон начинается с чистого ядра.
3. Внизу печатается сводка: строка для `experiments.csv` и конфиг. Её нужно скопировать в репозиторий
   (строку — в `experiments.csv`, конфиг — в `configs/<id>_<name>.json`).
4. После отправки сабмита на Kaggle впишите score в колонку `kaggle_score`.

Сабмит сохраняется как `submission_<name>.csv` и как `submission.csv`.

## 1. Подготовка

In [ ]:
import copy
import json
import random
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.transforms import v2

DATA_DIR = Path("/kaggle/input/datasets/oripyro/week-5")
# DATA_DIR = Path("week-5-competition-geo-satellites-classification")  # локально
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 4
MEAN = (0.485, 0.456, 0.406)      # нормализация ImageNet
STD = (0.229, 0.224, 0.225)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


print("device:", DEVICE)
print("data folder:", sorted(p.name for p in DATA_DIR.iterdir()))

## 2. Конфиг эксперимента
Всё, чем один прогон отличается от другого, задаётся здесь.

* `stages` — этапы обучения. В `train` перечислены блоки бэкбона, которые обучаются на этом этапе
  (остальные заморожены); классификатор обучается всегда. `"all"` — весь бэкбон, `[]` — только классификатор.
  Имена блоков: у resnet — `layer1`…`layer4`, у efficientnet / convnext — `features.0`…`features.7`.
* В конце каждого этапа модель возвращается к весам лучшей по val эпохи этого этапа.
* Аугментация с нулевой вероятностью/силой не добавляется вовсе.

In [ ]:
CFG = {
    "name": "r18_224_3stage",
    "notes": "перенос в новый ноутбук: повтор лучшего прогона (0.937)",
    "seed": 42,
    "img_size": 224,
    "batch_size": 128,
    "backbone": "resnet18",          # resnet18 | resnet50 | efficientnet_b0 | convnext_tiny
    "head_dropout": 0.3,
    "aug": {
        "hflip_p": 0.5,
        "vflip_p": 0.5,
        "rotation_angles": [0, 90, 180, 270],
        "brightness": 0.2,           # ColorJitter: ×(1-b) ... ×(1+b)
        "contrast": 0.2,
        "blur_kernel": 9,
        "blur_sigma": [0.5, 3.0],
        "blur_p": 0.3,
        "jpeg_quality": [10, 80],
        "jpeg_p": 0.5,
        "noise_sigmas": [0.02, 0.05, 0.1, 0.15, 0.2],
        "noise_p": 0.1,
        "gray_p": 0.0,
    },
    "stages": [
        {"train": ["layer3", "layer4"], "epochs": 5, "lr": 1e-4},
        {"train": ["layer4"], "epochs": 5, "lr": 1e-4},
        {"train": [], "epochs": 10, "lr": 1e-4},
    ],
}

## 3. Данные

In [ ]:
train_table = pd.read_csv(DATA_DIR / "train.csv", dtype={"id": str})
val_table = pd.read_csv(DATA_DIR / "val.csv", dtype={"id": str})
test_table = pd.read_csv(DATA_DIR / "test.csv", dtype={"id": str})
classes = pd.read_csv(DATA_DIR / "classes.csv").sort_values("label").class_name.tolist()
NUM_CLASSES = len(classes)

print(f"train: {len(train_table)} images | val: {len(val_table)} | test: {len(test_table)} | classes: {NUM_CLASSES}")
print(classes)


def image_file(folder, image_id):
    # Id — это имена файлов вроде '1000.jpg' (просто числа тоже подходят)
    name = image_id if image_id.endswith(".jpg") else f"{image_id}.jpg"
    return DATA_DIR / folder / name


_image_cache = {}                 # (папка, размер) -> список картинок; чтобы не читать диск повторно


def load_images(table, folder, img_size):
    key = (folder, img_size)
    if key not in _image_cache:
        images = []
        for image_id in tqdm(table.id, desc=f"loading {folder} @ {img_size}"):
            with Image.open(image_file(folder, image_id)) as img:
                images.append(img.convert("RGB").resize((img_size, img_size), Image.BOX))
        _image_cache[key] = images
    return _image_cache[key]


class ContestDataset(Dataset):
    def __init__(self, table, images, transform):
        self.ids = table.id.tolist()
        self.labels = table.label.tolist() if "label" in table.columns else [-1] * len(table)
        self.images = images
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return self.transform(self.images[i]), self.labels[i]

## 4. Аугментации

In [ ]:
def build_train_transform(aug):
    t = []
    if aug["hflip_p"] > 0:
        t.append(transforms.RandomHorizontalFlip(p=aug["hflip_p"]))
    if aug["vflip_p"] > 0:
        t.append(transforms.RandomVerticalFlip(p=aug["vflip_p"]))
    if aug["rotation_angles"]:
        t.append(transforms.RandomChoice([transforms.RandomRotation((a, a)) for a in aug["rotation_angles"]]))
    if aug["brightness"] > 0 or aug["contrast"] > 0:
        t.append(transforms.ColorJitter(brightness=aug["brightness"], contrast=aug["contrast"]))
    if aug["blur_p"] > 0:
        t.append(transforms.RandomApply([transforms.GaussianBlur(kernel_size=aug["blur_kernel"],
                                                                 sigma=aug["blur_sigma"])], p=aug["blur_p"]))
    if aug["jpeg_p"] > 0:
        t.append(transforms.RandomApply([v2.JPEG(quality=aug["jpeg_quality"])], p=aug["jpeg_p"]))
    if aug["noise_p"] > 0:
        t.append(transforms.RandomApply([v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.RandomChoice([v2.GaussianNoise(sigma=s) for s in aug["noise_sigmas"]]),
            v2.ToPILImage(),
        ])], p=aug["noise_p"]))
    if aug["gray_p"] > 0:
        t.append(transforms.RandomGrayscale(p=aug["gray_p"]))
    t += [transforms.ToTensor(), transforms.Normalize(MEAN, STD)]
    return transforms.Compose(t)


eval_transform = transforms.Compose([     # валидация и тест: никаких аугментаций
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

In [ ]:
# Проверка аугментаций: оригинал и несколько случайных вариантов train-трансформа
def show_augmentations(image_id="10.jpg", n=6, size=2.5):
    img = Image.open(image_file("train", image_id)).convert("RGB").resize((CFG["img_size"],) * 2, Image.BOX)
    tfm = build_train_transform(CFG["aug"])
    fig, axes = plt.subplots(1, n + 1, figsize=(size * (n + 1), size))
    axes[0].imshow(img); axes[0].set_title("original", fontsize=9)
    for ax in axes[1:]:
        x = tfm(img) * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]
        ax.imshow(x.clamp(0, 1).permute(1, 2, 0))
    for ax in axes:
        ax.axis("off")
    plt.tight_layout(); plt.show()


show_augmentations()

## 5. Модель

In [ ]:
# имя -> (конструктор, веса, путь к последнему Linear, который заменяем на Identity)
BACKBONES = {
    "resnet18": (models.resnet18, models.ResNet18_Weights.IMAGENET1K_V1, "fc"),
    "resnet50": (models.resnet50, models.ResNet50_Weights.IMAGENET1K_V2, "fc"),
    "efficientnet_b0": (models.efficientnet_b0, models.EfficientNet_B0_Weights.IMAGENET1K_V1, "classifier.1"),
    "convnext_tiny": (models.convnext_tiny, models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1, "classifier.2"),
}


def build_model(cfg):
    # Модель = nn.Sequential(бэкбон, классификатор): model[0] — бэкбон, model[1] — классификатор
    constructor, weights, head_path = BACKBONES[cfg["backbone"]]
    backbone = constructor(weights=weights)
    num_features = backbone.get_submodule(head_path).in_features
    parent_path, _, child = head_path.rpartition(".")
    setattr(backbone.get_submodule(parent_path), child, nn.Identity())
    classifier = nn.Sequential(
        nn.BatchNorm1d(num_features),
        nn.Dropout(cfg["head_dropout"]),
        nn.Linear(num_features, NUM_CLASSES),
    )
    return nn.Sequential(backbone, classifier)


def set_trainable(model, layers):
    backbone, classifier = model
    for p in backbone.parameters():
        p.requires_grad = layers == "all"
    if layers != "all":
        for name in layers:
            for p in backbone.get_submodule(name).parameters():
                p.requires_grad = True
    for p in classifier.parameters():
        p.requires_grad = True


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 6. Обучение и оценка
`train_stage` — прежний `train_model`: Adam, после каждой эпохи loss/accuracy на train и val,
в конце возврат к весам лучшей эпохи. `run_experiment` собирает всё вместе по `CFG`.

In [ ]:
criterion = nn.CrossEntropyLoss()
TRAIN_GENERATOR = torch.Generator()        # сид выставляет run_experiment


def make_loader(dataset, shuffle, batch_size):
    return DataLoader(
        dataset, batch_size=batch_size,
        shuffle=shuffle,
        generator=TRAIN_GENERATOR if shuffle else None,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=shuffle,
    )


@torch.no_grad()
def evaluate(model, dataset, batch_size):
    model.eval()
    total_loss, correct = 0.0, 0
    for images, labels in make_loader(dataset, shuffle=False, batch_size=batch_size):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        total_loss += criterion(outputs, labels).item() * len(images)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
    return total_loss / len(dataset), correct / len(dataset)


@torch.no_grad()
def predict(model, dataset, batch_size):
    model.eval()
    predictions = []
    for images, _ in make_loader(dataset, shuffle=False, batch_size=batch_size):
        outputs = model(images.to(DEVICE))
        predictions.append(outputs.argmax(dim=1).cpu())
    return torch.cat(predictions).numpy()


def train_stage(model, name, train_data, val_data, epochs, lr, batch_size):
    model = model.to(DEVICE)
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(trainable, lr=lr)
    train_loader = make_loader(train_data, shuffle=True, batch_size=batch_size)

    history, best_acc, best_weights = [], -1.0, None
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, correct = 0.0, 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(images)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
        train_loss, train_acc = total_loss / len(train_data), correct / len(train_data)

        val_loss, val_acc = evaluate(model, val_data, batch_size)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                        "train_acc": train_acc, "val_acc": val_acc})

        if val_acc > best_acc:
            best_acc, best_weights = val_acc, copy.deepcopy(model.state_dict())
        print(f"{name} | epoch {epoch:2d} | train loss {train_loss:.3f} | val loss {val_loss:.3f} | "
              f"train acc {train_acc:.3f} | val acc {val_acc:.3f}")

    model.load_state_dict(best_weights)
    print(f"{name}: best validation accuracy {best_acc:.3f}")
    return pd.DataFrame(history)

In [ ]:
def stages_str(stages):
    # Короткая запись этапов для таблицы: "layer3+layer4:5@1e-04 | head:10@1e-04"
    parts = []
    for s in stages:
        layers = "all" if s["train"] == "all" else ("+".join(s["train"]) or "head")
        parts.append(f"{layers}:{s['epochs']}@{s['lr']:.0e}")
    return " | ".join(parts)


def run_experiment(cfg):
    start = time.time()
    seed_everything(cfg["seed"])
    TRAIN_GENERATOR.manual_seed(cfg["seed"])

    size = cfg["img_size"]
    train_set = ContestDataset(train_table, load_images(train_table, "train", size), build_train_transform(cfg["aug"]))
    val_set = ContestDataset(val_table, load_images(val_table, "val", size), eval_transform)
    test_set = ContestDataset(test_table, load_images(test_table, "test", size), eval_transform)

    model = build_model(cfg)
    seed_everything(cfg["seed"])

    histories = []
    for i, stage in enumerate(cfg["stages"], 1):
        set_trainable(model, stage["train"])
        print(f"--- stage {i}: train {stage['train']} | trainable parameters: "
              f"{count_parameters(model)} of {sum(p.numel() for p in model.parameters())}")
        h = train_stage(model, f"stage {i}", train_set, val_set, stage["epochs"], stage["lr"], cfg["batch_size"])
        h["stage"] = i
        histories.append(h)
    history = pd.concat(histories, ignore_index=True)
    history["global_epoch"] = range(1, len(history) + 1)

    # итоговые веса — лучшая эпоха последнего этапа
    last = history[history.stage == len(cfg["stages"])]
    best_row = history.loc[last.val_acc.idxmax()]

    submission = pd.DataFrame({"id": test_set.ids, "label": predict(model, test_set, cfg["batch_size"])})
    assert submission.id.tolist() == test_table.id.tolist()
    assert submission.label.between(0, NUM_CLASSES - 1).all()
    submission.to_csv(OUT_DIR / f"submission_{cfg['name']}.csv", index=False)
    submission.to_csv(OUT_DIR / "submission.csv", index=False)

    summary = {
        "id": "",                                 # номер проставляется при переносе в репозиторий
        "date": datetime.now().strftime("%Y-%m-%d"),
        "name": cfg["name"],
        "backbone": cfg["backbone"],
        "img_size": cfg["img_size"],
        "stages": stages_str(cfg["stages"]),
        "best_val": round(float(best_row.val_acc), 4),
        "best_epoch": int(best_row.global_epoch),
        "time_min": round((time.time() - start) / 60, 1),
        "kaggle_score": "",
        "notes": cfg["notes"],
    }
    (OUT_DIR / "logs").mkdir(exist_ok=True)
    history.to_csv(OUT_DIR / "logs" / f"{cfg['name']}.csv", index=False)
    return model, history, summary

In [ ]:
def plot_history(history, name):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric in zip(axes, ["loss", "acc"]):
        ax.plot(history.global_epoch, history[f"train_{metric}"], label="training")
        ax.plot(history.global_epoch, history[f"val_{metric}"], label="validation")
        for boundary in history.groupby("stage").global_epoch.max().iloc[:-1]:
            ax.axvline(boundary + 0.5, color="gray", linestyle="--", linewidth=0.8)
        ax.set_title(f"{name}: {metric}"); ax.set_xlabel("epoch"); ax.legend()
    plt.tight_layout(); plt.show()

## 7. Запуск

In [ ]:
model, history, summary = run_experiment(CFG)

In [ ]:
plot_history(history, CFG["name"])

## 8. Сводка — скопируйте в репозиторий

In [ ]:
print("=== строка для experiments.csv ===")
print(pd.DataFrame([summary]).to_csv(index=False, header=False).strip())
print()
print("=== конфиг для configs/<id>_<name>.json ===")
print(json.dumps(CFG, ensure_ascii=False, indent=2))

In [ ]:
# Проверка генератора загрузчика: подряд идущие вызовы дают разный порядок, сброс воспроизводит его
train_check = ContestDataset(train_table, load_images(train_table, "train", CFG["img_size"]), eval_transform)


def first_batch_labels():
    _, labels = next(iter(make_loader(train_check, shuffle=True, batch_size=CFG["batch_size"])))
    return labels


TRAIN_GENERATOR.manual_seed(CFG["seed"])
labels_first = first_batch_labels()
labels_second = first_batch_labels()
TRAIN_GENERATOR.manual_seed(CFG["seed"])
labels_again = first_batch_labels()
assert not torch.equal(labels_first, labels_second)
assert torch.equal(labels_first, labels_again)
print("генератор загрузчика: ok")